# Cuál es el problema?

Abandono de clientes (Churn)

Trabajamos con una empresa de telecomunicaciones que quiere ***predecir cuales clientes van a abandonar el servicio*** (churn) para poder tomar acciones preventivas.

Por qué es importante?

* Conseguir un cliente nuevo representa entre 5x y 25x más que retener uno existente.
* Si podemos predecir quien se va, podemos ofrecerle un descuento o llamarlo antes de que cancele.



In [ ]:
# Instalar matplotlib si no está disponible
%pip install matplotlib --quiet

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split # Dividir datos en entrenamiento y prueba
from sklearn.compose import ColumnTransformer # Para transformar columnas específicas
from sklearn.preprocessing import OneHotEncoder # Para escalar y codificar variables
from sklearn.linear_model import LogisticRegression # Modelo de regresión logística

# Metricas de evaluación para la clasificación

from sklearn.metrics import (
    confusion_matrix, # Matriz de confusión
    classification_report, # reporte con precisión, recall y f1-score
    accuracy_score, # Porcentaje de predicciones correctas
)

import matplotlib.pyplot as plt

print("librerias importadas correctamente")


: 

In [9]:
#cargar el archivo CSV en un dataframe (tabla de datos)
df = pd.read_csv("abandono_clientes.csv")
print("Archivo CSV cargado correctamente")

# Mostrar las primeras 5 filas del dataframe para verificar que se cargó correctamente
print("Mostrando las primeras 5 filas del dataframe:")
print(df.head())

Archivo CSV cargado correctamente
Mostrando las primeras 5 filas del dataframe:
   antiguedad_meses  factura_mensual  llamadas_soporte   contrato  \
0                52            61.18                 0  Dos_anios   
1                15            54.89                 1    Mensual   
2                72           112.95                 3      Anual   
3                61           103.06                 0    Mensual   
4                21           116.50                 2    Mensual   

   satisfaccion  abandono  
0             3         0  
1             1         1  
2             2         0  
3             1         1  
4             1         1  


In [10]:
# Informacion general del dataset
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
print()

# Tipo de dato de cada columna
print("Tipos de datos:")
for col, dtype in df.dtypes.items():
    print(f"  {col:20s} -> {dtype}")

print()

# Verificar valores faltantes
nulos = df.isna().sum()
if nulos.sum() == 0:
    print("No hay valores faltantes en el dataset.")
else:
    print(f"ATENCION: hay {nulos.sum()} valores nulos.")
    print(nulos[nulos > 0])

Filas: 800, Columnas: 6

Tipos de datos:
  antiguedad_meses     -> int64
  factura_mensual      -> float64
  llamadas_soporte     -> int64
  contrato             -> object
  satisfaccion         -> int64
  abandono             -> int64

No hay valores faltantes en el dataset.


In [11]:
df.describe().round(2) #Estadísticas descriptivas de las columnas numéricas


,antiguedad_meses,factura_mensual,llamadas_soporte,satisfaccion,abandono
count,800.00,800.00,800.00,800.00,800.00
mean,35.53,70.65,2.02,2.97,0.35
std,21.11,29.24,1.42,1.43,0.48
min,1.00,20.46,0.00,1.00,0.00
25%,17.00,43.86,1.00,2.00,0.00
50%,35.00,71.82,2.00,3.00,0.00
75%,54.00,96.10,3.00,4.00,1.00
max,72.00,119.94,7.00,5.00,1.00


In [12]:
# DISTRIBUCION DE LA VARIABLE OBJETIVO

#SI EL 99% DE LOS CLIENTES NO ABANDONAN, ES UN MODELO QUE TENDRIA UNA PRESICION (ACCURACY) COMPLETAMENTE INUTIL
# PORQUE SIEMPRE PREDECERIA EL 99% DE LOS DATOS, POR TANTO HAY QUIE BALANCEAR LAS CLASES.

conteo=df["abandono"].value_counts()
porcentaje=df["abandono"].value_counts(normalize=True)*100

print("distribucion de la variable objetivo (Abandono):")
print(f" Permanece: (0): {conteo[0]} clientes, {porcentaje[0]:.1f}%")
print(f" Abandona: (1): {conteo[1]} clientes, {porcentaje[1]:.1f}%")
print()
print("las clases estan moderadamente desbalanceadas")
print("hay mas clientes que permanecen, que clientes que abandonan")
print("esto es lo tipico en la realidad: la mayoria de clientes NO se van")


distribucion de la variable objetivo (Abandono):
 Permanece: (0): 517 clientes, 64.6%
 Abandona: (1): 283 clientes, 35.4%

las clases estan moderadamente desbalanceadas
hay mas clientes que permanecen, que clientes que abandonan
esto es lo tipico en la realidad: la mayoria de clientes NO se van


### Analisis exploratocio: como se comportan las variables segun abandono?###


In [13]:
comparacion = df.groupby("abandono")[["antiguedad_meses", "factura_mensual",
                                       "llamadas_soporte", "satisfaccion"]].mean().round(2)

comparacion.index = ["Permanece (0)", "Abandona (1)"]

print("Promedio de cada variable segun abandono:")
print()
comparacion

Promedio de cada variable segun abandono:



,antiguedad_meses,factura_mensual,llamadas_soporte,satisfaccion
Permanece (0),39.43,67.02,1.88,3.14
Abandona (1),28.41,77.27,2.28,2.67


In [16]:
#analisis del tipo de contrato vs abandono

# El dataset tiene la columna 'contrato' (no 'tipo_contrato'), usar el nombre correcto
tabla_contrato = pd.crosstab(
    df["contrato"], 
    df["abandono"], 
    normalize="index"
).round(3)*100

tabla_contrato.columns = ["Permanece (%)", "Abandona (%)"]
print("porcentaje de abandono segun tipo de contrato:")
tabla_contrato

porcentaje de abandono segun tipo de contrato:


,Permanece (%),Abandona (%)
contrato,,
Anual,74.0,26.0
Dos_anios,77.3,22.7
Mensual,54.0,46.0


# Preparar los datos para el modelo

1. Separar variables predictivas (X) de la variable objetivo
2. entrenar los datos con el 80% del dataset y probar con el 20% del dataset



In [17]:
#Paso 1: Separar variables predictivas (X) de la variable objetivo (y)
X = df[["antiguedad_meses", "factura_mensual", "llamadas_soporte", "satisfaccion", "contrato"]]

y = df["abandono"]

#Paso 2: Dividir el dataset en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Datos de entrenamiento: {X_train.shape[0]} clientes")
print(f"Datos de prueba: {X_test.shape[0]} clientes")

print("\n")
print(f"preparacion de abandono en entrenamiento: {y_train.mean()*100:.1f}%")
print(f"preparacion de abandono en prueba: {y_test.mean()*100:.1f}%")
print()
print("la proporciones son similares gracias al promedio estratificado (stratify=y) en train_test_split")


Datos de entrenamiento: 640 clientes
Datos de prueba: 160 clientes


preparacion de abandono en entrenamiento: 35.3%
preparacion de abandono en prueba: 35.6%

la proporciones son similares gracias al promedio estratificado (stratify=y) en train_test_split


# Construir el Pipeline y entrenar el modelo

1. Calcula una puntuacion lineal
2. pasa por esa puntuacion qque la convierte en una probabilidad entre 0 y 1
3. si la probabilidad es mayoer a 0.5, predice la clase 1 (abandona): si no, predice 0 (permanece)

In [20]:
from sklearn.pipeline import Pipeline

numeric_features = ["antiguedad_meses", "factura_mensual",
                    "llamadas_soporte", "satisfaccion"]
categorical_features = ["contrato"]


preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)


pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000)),
])

# Entrenar el modelo con los datos de entrenamiento
pipe.fit(X_train, y_train)

print("Modelo entrenado correctamente")

Modelo entrenado correctamente


# Hacer predicciones

usamos el modelo entrenado para predecir el abandono en los datos de prueba


In [21]:
# Generar predicciones en los datos de prueba
y_pred = pipe.predict(X_test)

# Mostrar las primeras 10 predicciones
comparacion_pred = pd.DataFrame({
    "Real": y_test.values[:10],
    "Predicción": y_pred[:10],
    "correcta": ["si" if r== p else "no" for r, p in zip(y_test.values[:10], y_pred[:10])]
})

print("las primeras 10 predicciones vs valores reales:")
comparacion_pred

las primeras 10 predicciones vs valores reales:


,Real,Predicción,correcta
0,1,1,si
1,0,0,si
2,0,0,si
3,0,0,si
4,0,0,si
5,0,0,si
6,0,0,si
7,0,0,si
8,0,0,si
9,0,0,si


# Evaluacion del modelo: Exactitud/Presicion (Acurracy) 

Cuando hablamos del porcentaje de preciosion nos referimos a las predicciones que fueron correctas

Acurracy = predicciones correctas/Total de predicciones

Ejemplo: si el 95% de clientes no abandona, es un modelo que siempre diga "NO abandona" tendria un 95% de precision, pero jamas detectaria a un cliente en riesgo, Seria inutil.


In [23]:
acc = accuracy_score(y_test, y_pred)
print(f"Exactitud (Accuracy) del modelo: {acc:.4f} ({acc*100:.1f}%)")
print()
print(f"Esto significa que el modelo acertó {int(acc*len(y_test))} de {len(y_test)} predicciones.")

Exactitud (Accuracy) del modelo: 0.7625 (76.2%)

Esto significa que el modelo acertó 122 de 160 predicciones.
